In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('/Users/professorx/Projects/Capstone/project_work/data/raw/diabetic_data.csv',na_values="?")

In [3]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


# Data cleaning

In [4]:
df.shape

(101766, 50)

In [5]:
# Duplicates handling

df = df.drop_duplicates(
    subset='patient_nbr',
    keep = "first"
)

df.shape

(71518, 50)

In [6]:
# Invalid discharge records handling
invalid = [11,13,14,19,20,21]

df = df[~df['discharge_disposition_id'].isin(invalid)]

df.shape

(69973, 50)

In [7]:
# No use columns
df.drop(columns=['weight'],inplace=True)
print('Dropped..')
df.shape

Dropped..


(69973, 49)

In [8]:
# constant columns handling no use for models

constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
print(constant_cols)

df.drop(columns=constant_cols,inplace=True)
df.shape

['examide', 'citoglipton', 'glimepiride-pioglitazone']


(69973, 46)

# Create Target Variable

In [9]:
df['target'] = (df['readmitted'] == '<30').astype(int)
df['target'].value_counts(normalize=True) * 100

target
0    91.029397
1     8.970603
Name: proportion, dtype: float64

In [10]:
# Dropping readmitted column

df.drop(columns=['readmitted'],inplace=True)
print("Dropped..")

Dropped..


# Feature Engineering

In [11]:
age_mapping = {
    "[0-10)":5,
    "[10-20)":15,
    "[20-30)":25,
    "[30-40)":35,
    "[40-50)":45,
    "[50-60)":55,
    "[60-70)":65,
    "[70-80)":75,
    "[80-90)":85,
    "[90-100)":95
}

df['mid_age'] = df['age'].map(age_mapping)

In [12]:
# Service works

df["service_utilization"] = (
    df["number_outpatient"]
    + df["number_emergency"]
    + df["number_inpatient"]
)

In [13]:
# Working on Medication columns
medication_cols = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]

# counting medications that are not -> NO -> similar to like Steady UP anything
df['medication_change_count'] = df[medication_cols].ne('No').sum(axis=1)
df['medication_change_count'].describe()

count    69973.00000
mean         1.19013
std          0.94253
min          0.00000
25%          1.00000
50%          1.00000
75%          2.00000
max          6.00000
Name: medication_change_count, dtype: float64

In [14]:
# Generalizing ICD-9 codes 

def group_icd9(code):
    """
    Group ICD-9 diagnosis codes into standard ICD-9 chapters.
    """

    if pd.isna(code):
        return "Unknown"

    code = str(code).strip()

    if code.startswith("V"):
        return "Immunizations/Screening"

    if code.startswith("E"):
        return "External Causes/Accidents"

    try:
        code = float(code)
    except ValueError:
        return "Other"

    if str(code).startswith("250"):
        return "Diabetes"

    elif 1 <= code <= 139:
        return "Infectious & Parasitic"

    elif 140 <= code <= 239:
        return "Neoplasms"

    elif 240 <= code <= 279:
        return "Endocrine/Metabolic"

    elif 280 <= code <= 289:
        return "Blood Disorders"

    elif 290 <= code <= 319:
        return "Mental Disorders"

    elif 320 <= code <= 389:
        return "Nervous System"

    elif 390 <= code <= 459:
        return "Circulatory"

    elif 460 <= code <= 519:
        return "Respiratory"

    elif 520 <= code <= 579:
        return "Digestive"

    elif 580 <= code <= 629:
        return "Genitourinary"

    elif 630 <= code <= 677:
        return "Pregnancy"

    elif 680 <= code <= 709:
        return "Skin"

    elif 710 <= code <= 739:
        return "Musculoskeletal"

    elif 740 <= code <= 759:
        return "Congenital"

    elif 760 <= code <= 779:
        return "Perinatal"

    elif 780 <= code <= 799:
        return "Symptoms"

    elif 800 <= code <= 999:
        return "Injury"

    return "Other"

In [15]:
diag_cols = ['diag_1','diag_2','diag_3']

for col in diag_cols:
    df[f'{col}_group'] = df[col].apply(group_icd9)

In [16]:
df['diag_1_group'].value_counts()

diag_1_group
Circulatory                  21316
Respiratory                   6446
Digestive                     6325
Diabetes                      5748
Symptoms                      5503
Injury                        4694
Musculoskeletal               4064
Genitourinary                 3414
Neoplasms                     2538
Endocrine/Metabolic           1850
Skin                          1780
Infectious & Parasitic        1685
Mental Disorders              1545
Immunizations/Screening        918
Nervous System                 858
Blood Disorders                651
Pregnancy                      586
Congenital                      41
Unknown                         10
External Causes/Accidents        1
Name: count, dtype: int64

In [17]:
df['diag_2_group'].value_counts()

diag_2_group
Circulatory                  21782
Diabetes                      9700
Respiratory                   6445
Endocrine/Metabolic           5605
Genitourinary                 5042
Symptoms                      3168
Digestive                     2704
Skin                          2228
Blood Disorders               2075
Mental Disorders              1856
Injury                        1824
Neoplasms                     1599
Musculoskeletal               1295
Infectious & Parasitic        1240
Immunizations/Screening       1217
Nervous System                 894
External Causes/Accidents      570
Pregnancy                      353
Unknown                        293
Congenital                      83
Name: count, dtype: int64

In [18]:
df.drop(columns=["diag_1", "diag_2", "diag_3"], inplace=True)

In [19]:
# final df size
df.shape

(69973, 49)

# Feature selection

In [20]:
# Dropping encounter_id and patient_nbr
df.drop(columns=['encounter_id','patient_nbr'],inplace=True)
print("Dropped..")

Dropped..


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69973 entries, 0 to 101765
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   race                      68055 non-null  object
 1   gender                    69973 non-null  object
 2   age                       69973 non-null  object
 3   admission_type_id         69973 non-null  int64 
 4   discharge_disposition_id  69973 non-null  int64 
 5   admission_source_id       69973 non-null  int64 
 6   time_in_hospital          69973 non-null  int64 
 7   payer_code                39558 non-null  object
 8   medical_specialty         36334 non-null  object
 9   num_lab_procedures        69973 non-null  int64 
 10  num_procedures            69973 non-null  int64 
 11  num_medications           69973 non-null  int64 
 12  number_outpatient         69973 non-null  int64 
 13  number_emergency          69973 non-null  int64 
 14  number_inpatient          

In [22]:
X = df.drop(columns='target')
y = df['target']
print(X.shape,y.shape)

(69973, 46) (69973,)


In [23]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print(numeric_features)
print(categorical_features)

['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'mid_age', 'service_utilization', 'medication_change_count']
['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']


# Train Test Split

In [24]:
# Three data split -> train(70%),test(15%),validation(15%)

X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

In [25]:
X_val,X_test,y_val,y_test = train_test_split(X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42)

In [26]:
print(X_train.shape,X_test.shape,X_val.shape)
print(y_train.shape,y_test.shape,y_val.shape)

(48981, 46) (10496, 46) (10496, 46)
(48981,) (10496,) (10496,)


# Preprocessing Pipeline

In [27]:
numerical_pipeline = Pipeline(
    [
        ('imputer',SimpleImputer(strategy='median')),
        ('scalar',StandardScaler())
    ]
)

In [28]:
categorical_pipeline = Pipeline(
    [
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [29]:
# final pipeline

preprocessor_pipeline = ColumnTransformer([
    ('numeric',numerical_pipeline,numeric_features),
    ('categoric',categorical_pipeline,categorical_features)
])

# Save Artifacts

In [30]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)